# Step1B: Preprocessing Audit

Advanced notebook template for preprocessing after QC.

This notebook starts from `Step1-sce_cleaned.h5ad` produced by Step1A and focuses on the preprocessing benchmark module:

- QC handoff context
- gene biotype annotation and optional filtering
- normalization and `.raw` setup
- HVG selection and stability checks
- regression, scaling, PCA, integration decision, and final graph
- layer transitions, per-step evidence, module maturity, and compact audit output

Input: `data/processed/Step1-sce_cleaned.h5ad`.  
Output: `data/processed/Step2-sce_preprocessed.h5ad`.


## Environment Setup

In [ ]:
import gc
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import scLucid as scl

warnings.filterwarnings('ignore')

scl.setup_logging('INFO')
scl.set_figure_params(
    dpi=150,
    dpi_save=300,
    figsize=(6, 5),
    style='seaborn-v0_8',
    style_dict={'axes.grid': True},
    color_theme='default',
)


## Parameter Configuration**Edit this cell when switching projects.**All paths, sample names, metadata, and analysis options are centralized here.

In [ ]:
# ==============================================================================
# PROJECT PATHS
# ==============================================================================
# Set BASE_DIR to your project root. All other paths are relative to it.
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = BASE_DIR / 'data/RAW'            # 10X output directories (one per sample)
DATA_DIR = BASE_DIR / 'data/processed'     # Where intermediate .h5ad files are saved
RESULTS_DIR = BASE_DIR / 'results'         # All QC/preprocessing outputs go here
SAMPLE_METADATA_PATH = BASE_DIR / 'sample_metadata.csv'
SOURCE_NOTEBOOK_NAME = 'Step1B-Preprocessing_Audit.ipynb'
WORKFLOW_LAYER = 'advanced_notebook_api'

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ==============================================================================
# SPECIES & TISSUE
# ==============================================================================
SPECIES = 'mouse'               # 'human' or 'mouse'
TISSUE = None                   # Tissue context hint, e.g. 'tumor', 'blood', 'brain'
TISSUE_TYPE = 'tumor_tissue'    # 'tumor_tissue', 'pbmc_or_blood', 'cell_line', 'organoid', 'unknown'

# ==============================================================================
# SAMPLES & METADATA
# ==============================================================================
# Preferred: provide sample_metadata.csv with at least SAMPLE_KEY.
# Optional columns such as GROUP_KEY and BATCH_KEY are attached to adata.obs.
# If you do not use a metadata CSV, set ALL_SAMPLES explicitly and optionally fill GROUP_DICT/BATCH_DICT.
SAMPLE_KEY = 'sampleID'
ALL_SAMPLES = None              # None = derive from sample_metadata.csv; otherwise ['sample_1', 'sample_2']
GROUP_KEY = 'group'             # Set to None if no biological group metadata is available.
BATCH_KEY = 'batch'             # Set to None if no technical batch metadata is available.
GROUP_DICT = {}                 # Optional fallback: {'sample_1': 'control'}
BATCH_DICT = {}                 # Optional fallback: {'sample_1': 'batch_1'}
BIOLOGY_COLUMNS = [GROUP_KEY] if GROUP_KEY else []

# Optional custom colors. Leave empty for automatic Scanpy palettes.
SAMPLE_COLORS = {}
GROUP_COLORS = {}
BATCH_COLORS = {}

# ==============================================================================
# QC OPTIONS
# ==============================================================================
DOUBLET_METHOD = 'doubletdetection'  # 'scrublet', 'doubletdetection', or 'solo'
THRESHOLD_METHOD = 'mad'             # 'mad', 'gmm', or 'percentile'
MAD_MULTIPLIERS = [3, 4, 5]
MAX_CELLS_FOR_PLOTTING = 100_000

# Adaptive marking
ADAPTIVE_BATCH_KEY = SAMPLE_KEY
ADAPTIVE_METRICS = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']
ADAPTIVE_METHOD = 'hierarchical'

# Manual threshold overrides.
# Floors protect against overly permissive data-driven lower bounds; ceilings should be used sparingly.
MANUAL_MIN_GENES = 300
MANUAL_MIN_COUNTS = None
MANUAL_MAX_GENES = None
MANUAL_MAX_COUNTS = None
MANUAL_PC_MT = None
MANUAL_NMADS = 5.0
PC_TOP_GENES_THRESHOLD = None        # e.g. 50.0 to enable fixed pc_top_20_genes filtering

CRITERIA_TO_FILTER = [
    'outlier_min_genes',
    'outlier_qc_metrics',
    'outlier_n_genes_by_counts_adaptive',
    'outlier_total_counts_adaptive',
    'outlier_mt',
    # 'outlier_pct_counts_mt_adaptive',
    'predicted_doublet',
]
FILTER_COMBINATION = 'any'

# ==============================================================================
# PREPROCESSING OPTIONS
# ==============================================================================
RUN_BIOTYPE_ANNOTATION = True
BIOTYPE_REFERENCE_PATH = None
BIOTYPE_METHOD = 'reference'         # 'reference' uses scLucid bundled reference; 'custom' requires BIOTYPE_REFERENCE_PATH
KEEP_BIOTYPES = ['protein_coding']
FILTER_BIOTYPE_AFTER_RAW = True

NORM_METHOD = 'standard'
NORM_TARGET_SUM = 1e4
EXCLUDE_HIGHLY_EXPRESSED = True
USE_QUALITY_AWARE_NORM = False

HVG_N_TOP_GENES_SCANPY = 2500
HVG_N_TOP_GENES_CUSTOM = 1500
HVG_FLAVOR = 'seurat'
HVG_KEY_SCANPY = f'highly_variable_scanpy_{HVG_FLAVOR}'
HVG_KEY_CUSTOM = 'highly_variable_custom'
HVG_UNION_MODE = 'union'
HVG_MIN_N_SAMPLES = 2
RUN_HVG_STABILITY = True

VARS_TO_REGRESS = ['total_counts', 'pct_counts_mt', 'S_score', 'G2M_score']
REGRESS_IN_SCALE = False
SCALE_MAX_VALUE = 10
N_PCS_MAX = 50

N_NEIGHBORS_LIST = [15, 20, 25, 30]
N_PCS_LIST = [20, 25, 30, 35, 40, 45, 50]
N_JOBS = -1

RUN_INTEGRATION = 'auto'             # True, False, or 'auto'
INTEGRATION_METHOD = 'harmony'
INTEGRATION_BATCH_KEY = BATCH_KEY or SAMPLE_KEY
EVALUATE_INTEGRATION = True
RUN_EMBEDDING_OPTIMIZATION = True

# ==============================================================================
# CHECKPOINT, DIAGNOSTICS & REPORTING
# ==============================================================================
USE_CHECKPOINTS = False
CHECKPOINT_DIR = RESULTS_DIR / 'checkpoints'
SHOW_NORMALIZATION_DIAGNOSTICS = True
SHOW_SCALING_DIAGNOSTICS = True
SHOW_INTEGRATION_DIAGNOSTICS = True
GENERATE_HTML_REPORT = False
VALIDATE_STAGES = ['qc', 'preprocess']

print('Configuration loaded.')
print(f'  Project: {BASE_DIR}')
print(f'  Species: {SPECIES}')
print(f'  Tissue type: {TISSUE_TYPE}')
print(f'  Sample metadata: {SAMPLE_METADATA_PATH}')


## Helper FunctionsThese functions handle common operations: metadata merging, sample key resolution,integration auto-detection, and filtering audit.

In [ ]:
def load_sample_metadata():
    """Load optional sample metadata and normalize sample identifiers to strings."""
    if SAMPLE_METADATA_PATH and Path(SAMPLE_METADATA_PATH).exists():
        sep = '\t' if Path(SAMPLE_METADATA_PATH).suffix.lower() in {'.tsv', '.txt'} else ','
        meta = pd.read_csv(SAMPLE_METADATA_PATH, sep=sep)
        if SAMPLE_KEY not in meta.columns:
            raise ValueError(f"sample metadata must contain SAMPLE_KEY column: {SAMPLE_KEY!r}")
        meta = meta.copy()
        meta[SAMPLE_KEY] = meta[SAMPLE_KEY].astype(str)
        if meta[SAMPLE_KEY].duplicated().any():
            dup = meta.loc[meta[SAMPLE_KEY].duplicated(), SAMPLE_KEY].tolist()
            raise ValueError(f"sample metadata contains duplicate samples: {dup}")
        return meta
    return None


def resolve_samples(sample_metadata=None):
    """Resolve sample list from config or metadata."""
    if ALL_SAMPLES is not None:
        return [str(s) for s in ALL_SAMPLES]
    if sample_metadata is not None:
        return sample_metadata[SAMPLE_KEY].tolist()
    raise ValueError(
        'No samples configured. Provide sample_metadata.csv or set ALL_SAMPLES explicitly.'
    )


def build_metadata_dicts(sample_metadata=None):
    """Build metadata dicts for load_10x_data from metadata CSV and optional fallback dicts."""
    metadata_dicts = {}
    if sample_metadata is not None:
        for col in sample_metadata.columns:
            if col == SAMPLE_KEY:
                continue
            if sample_metadata[col].notna().any():
                metadata_dicts[col] = dict(zip(sample_metadata[SAMPLE_KEY], sample_metadata[col]))
    if GROUP_DICT and GROUP_KEY:
        metadata_dicts[GROUP_KEY] = {str(k): v for k, v in GROUP_DICT.items()}
    if BATCH_DICT and BATCH_KEY:
        metadata_dicts[BATCH_KEY] = {str(k): v for k, v in BATCH_DICT.items()}
    return metadata_dicts


def validate_project_config(samples, metadata_dicts):
    """Fail early on template configuration issues that otherwise cause silent provenance debt."""
    if not samples:
        raise ValueError('No samples resolved from configuration.')
    missing_dirs = [s for s in samples if not (RAW_DIR / s).exists()]
    if missing_dirs:
        print(f"WARNING: {len(missing_dirs)} sample directories were not found under RAW_DIR.")
        print(f"  First missing entries: {missing_dirs[:5]}")
    for required_key in [GROUP_KEY, BATCH_KEY]:
        if required_key and required_key in metadata_dicts:
            missing = [s for s in samples if s not in metadata_dicts[required_key]]
            if missing:
                raise ValueError(f"metadata for {required_key!r} is missing samples: {missing}")
    if INTEGRATION_BATCH_KEY and INTEGRATION_BATCH_KEY in BIOLOGY_COLUMNS:
        print(
            f"WARNING: INTEGRATION_BATCH_KEY={INTEGRATION_BATCH_KEY!r} is also listed as biology. "
            'Auto integration will likely skip correction to avoid removing biology.'
        )


def print_sample_crosstab(adata, group_key=None):
    """Print a crosstab of samples x groups for a quick data overview."""
    if group_key and group_key in adata.obs.columns:
        ctab = pd.crosstab(adata.obs[SAMPLE_KEY], adata.obs[group_key], margins=True)
    else:
        ctab = pd.DataFrame(adata.obs[SAMPLE_KEY].value_counts())
    print(ctab)


def audit_filtering(adata_before, adata_after, group_key=None):
    """Compare cell counts before and after filtering, by sample and optionally by group."""
    before = adata_before.obs.groupby(SAMPLE_KEY, observed=True).size()
    after = adata_after.obs.groupby(SAMPLE_KEY, observed=True).size()
    audit = pd.DataFrame({'before': before, 'after': after}).fillna(0).astype(int)
    audit['removed'] = audit['before'] - audit['after']
    audit['retention_rate'] = (audit['after'] / audit['before']).round(3)
    print('=== Retention by sample ===')
    display(audit.reset_index())

    audit_g = None
    if group_key and group_key in adata_before.obs.columns:
        before_g = adata_before.obs.groupby(group_key, observed=True).size()
        after_g = adata_after.obs.groupby(group_key, observed=True).size()
        audit_g = pd.DataFrame({'before': before_g, 'after': after_g}).fillna(0).astype(int)
        audit_g['removed'] = audit_g['before'] - audit_g['after']
        audit_g['retention_rate'] = (audit_g['after'] / audit_g['before']).round(3)
        print('=== Retention by group ===')
        display(audit_g.reset_index())
    return {'sample': audit, 'group': audit_g}


def audit_doublets(adata):
    """Check remaining doublet calls after filtering."""
    for col in ['predicted_doublet', 'scrublet_predicted', 'doubletdetection_predicted', 'heuristic_predicted']:
        if col in adata.obs.columns:
            n = adata.obs[col].sum()
            print(f'  {col}: {n} cells ({n / adata.n_obs * 100:.2f}%)')
            if col == 'predicted_doublet' and n > 0:
                print(f'    NOTE: {n} predicted doublets remain after filtering; review FilterConfig criteria.')


def detect_integration_confounding(adata, batch_key, biology_columns):
    """Check whether batch_key is confounded with biology columns."""
    if not batch_key or batch_key not in adata.obs.columns:
        return ['batch_key_missing']
    problems = []
    for col in biology_columns:
        if not col or col not in adata.obs.columns:
            continue
        mapping = adata.obs[[batch_key, col]].dropna().drop_duplicates()
        per_batch = mapping.groupby(batch_key)[col].nunique()
        if len(per_batch) > 0 and (per_batch <= 1).all():
            n_batch = adata.obs[batch_key].nunique()
            n_col = adata.obs[col].nunique()
            if n_batch == n_col:
                problems.append(f'{batch_key} is one-to-one with {col}; integration would remove biological signal')
    return problems


def resolve_integration_flag(adata, batch_key, biology_columns):
    """Determine whether to run integration, with auto-detection support."""
    if RUN_INTEGRATION is True:
        return True, []
    if RUN_INTEGRATION is False:
        return False, []
    if not batch_key or batch_key not in adata.obs.columns or adata.obs[batch_key].nunique() < 2:
        return False, ['single batch/sample; no integration needed']
    problems = detect_integration_confounding(adata, batch_key, biology_columns)
    return len(problems) == 0, problems


def build_color_palette(keys):
    """Build a combined color palette from configured label-color dictionaries."""
    palette = {}
    for k in keys:
        if k == SAMPLE_KEY:
            palette.update(SAMPLE_COLORS)
        elif GROUP_KEY and k == GROUP_KEY:
            palette.update(GROUP_COLORS)
        elif BATCH_KEY and k == BATCH_KEY:
            palette.update(BATCH_COLORS)
    return palette if palette else None


def build_config_snapshot():
    """Collect notebook parameters into a serializable config snapshot."""
    return {
        'project': {
            'base_dir': str(BASE_DIR),
            'raw_dir': str(RAW_DIR),
            'data_dir': str(DATA_DIR),
            'results_dir': str(RESULTS_DIR),
            'sample_metadata_path': str(SAMPLE_METADATA_PATH),
            'source_notebook': SOURCE_NOTEBOOK_NAME,
            'workflow_layer': WORKFLOW_LAYER,
        },
        'metadata': {
            'sample_key': SAMPLE_KEY,
            'group_key': GROUP_KEY,
            'batch_key': BATCH_KEY,
            'biology_columns': [c for c in BIOLOGY_COLUMNS if c],
        },
        'qc': {
            'doublet_method': DOUBLET_METHOD,
            'threshold_method': THRESHOLD_METHOD,
            'mad_multipliers': MAD_MULTIPLIERS,
            'adaptive_batch_key': ADAPTIVE_BATCH_KEY,
            'adaptive_metrics': ADAPTIVE_METRICS,
            'adaptive_method': ADAPTIVE_METHOD,
            'manual_min_genes': MANUAL_MIN_GENES,
            'manual_min_counts': MANUAL_MIN_COUNTS,
            'manual_max_genes': MANUAL_MAX_GENES,
            'manual_max_counts': MANUAL_MAX_COUNTS,
            'manual_pc_mt': MANUAL_PC_MT,
            'manual_nmads': MANUAL_NMADS,
            'pc_top_genes_threshold': PC_TOP_GENES_THRESHOLD,
            'criteria_to_filter': CRITERIA_TO_FILTER,
            'filter_combination': FILTER_COMBINATION,
            'tissue_type': TISSUE_TYPE,
            'species': SPECIES,
        },
        'preprocess': {
            'biotype_method': BIOTYPE_METHOD,
            'keep_biotypes': KEEP_BIOTYPES,
            'norm_method': NORM_METHOD,
            'norm_target_sum': NORM_TARGET_SUM,
            'exclude_highly_expressed': EXCLUDE_HIGHLY_EXPRESSED,
            'use_quality_aware_norm': USE_QUALITY_AWARE_NORM,
            'hvg_key_scanpy': HVG_KEY_SCANPY,
            'hvg_key_custom': HVG_KEY_CUSTOM,
            'hvg_union_mode': HVG_UNION_MODE,
            'vars_to_regress': VARS_TO_REGRESS,
            'n_pcs_max': N_PCS_MAX,
            'run_integration': RUN_INTEGRATION,
            'integration_method': INTEGRATION_METHOD,
            'integration_batch_key': INTEGRATION_BATCH_KEY,
        },
    }


def finalize_manual_review_summary(adata, module, workflow_name, steps, config, summary, save_dir=None):
    """Write the shared scLucid review contract for an advanced notebook workflow.

    The canonical review summary is a flat envelope matching workflow runs.
    `normalize_review_summary()` also adds a legacy `data` view so older
    notebooks/tests can read `review_summary['data']` during the schema
    transition.
    """
    from scLucid.utils import (
        build_config_lineage,
        export_review_summary,
        normalize_review_summary,
        record_config_lineage,
        save_result,
        save_workflow_result,
        validate_review_summary_schema,
    )

    inherited_context = {
        'source': 'notebook',
        'source_notebook': SOURCE_NOTEBOOK_NAME,
        'workflow_layer': WORKFLOW_LAYER,
        'species': SPECIES,
        'tissue': TISSUE,
        'tissue_type': TISSUE_TYPE,
        'sample_key': SAMPLE_KEY,
        'batch_key': BATCH_KEY,
        'group_key': GROUP_KEY,
    }
    lineage = build_config_lineage(
        inherited=inherited_context,
        stage_config=config,
        effective_config=config,
    )
    lineage['notebook'] = {
        'source_notebook': SOURCE_NOTEBOOK_NAME,
        'workflow_layer': WORKFLOW_LAYER,
        'config_snapshot': build_config_snapshot(),
    }
    review_summary = normalize_review_summary(
        summary,
        module=module,
        workflow_name=workflow_name,
        adata=adata,
        steps_executed=steps,
        config=config,
        config_lineage=lineage,
        artifacts={},
    )
    validate_review_summary_schema(review_summary, module=module, raise_on_error=True)
    save_result(adata, module, 'review_summary', review_summary)
    save_workflow_result(adata, module, workflow_name, steps, config)
    record_config_lineage(adata, module, lineage)
    if save_dir is not None:
        export_review_summary(review_summary, save_dir, module, adata=adata)
    return review_summary


print('Helper functions defined.')


---## **Step 3: Preprocessing**### Preprocessing strategy overview1. **Gene biotype annotation** — classify genes as protein_coding, lncRNA, etc.2. **Normalization** — log-normalize to 10K counts, store `.raw` for downstream DE3. **Normalization diagnostics** — before/after plots to verify normalization effect4. **Biotype filtering** — restrict to protein-coding (optional; `.raw` retains all genes)5. **HVG selection** — identify informative genes with two complementary methods6. **HVG stability** — bootstrap evaluation of HVG set robustness7. **Regression & scaling** — regress technical covariates, scale to unit variance8. **Scaling diagnostics** — verify regression and scaling effects9. **Diagnostic embedding** — pre-integration UMAP to assess sample-driven structure10. **Integration decision** — evaluate whether batch correction is needed11. **Integration evaluation** — post-hoc metrics (iLISI, ASW, kBet)12. **Final neighbors & UMAP** — optimized embedding on the final representation

In [ ]:
# Load QC-filtered data.
adata = sc.read_h5ad(str(DATA_DIR / 'Step1-sce_cleaned.h5ad'))
print(f'Dataset loaded: {adata.n_obs:,} cells, {adata.n_vars:,} genes')

# Ensure counts layer exists for normalization. QC should preserve raw counts in X.
if 'counts' not in adata.layers:
    adata.layers['counts'] = adata.X.copy()
    print("Created adata.layers['counts'] from adata.X. Verify X contains raw counts before reusing this template.")
else:
    print("Using existing adata.layers['counts'].")

print(f"\n{'=' * 60}")
print('Preprocessing')
print(f"{'=' * 60}")


### 3.1 Gene Biotype AnnotationAnnotate genes by biotype (protein_coding, lncRNA, pseudogene, etc.).Two modes:- **reference**: use scLucid's bundled reference (Ensembl)- **custom**: provide a gene_info file with `gene_name` and `biotype` columnsAnnotation is done **before** filtering so that `.raw` can retain all genes.

In [ ]:
if RUN_BIOTYPE_ANNOTATION:
    if BIOTYPE_METHOD == 'custom' and BIOTYPE_REFERENCE_PATH is not None:
        gene_info = pd.read_csv(BIOTYPE_REFERENCE_PATH, sep='\t')
        biotype_df = (
            gene_info[['external_gene_name', 'gene_biotype']]
            .dropna()
            .drop_duplicates()
            .rename(
                columns={
                    'external_gene_name': 'gene_name',
                    'gene_biotype': 'biotype',
                }
            )
        )
        adata = scl.pp.annotate_gene_biotypes(
            adata,
            biotype_df=biotype_df,
            method='custom',
            fuzzy_match=True,
            overwrite=True,
        )
    else:
        adata = scl.pp.apply_gene_biotype_strategy(
            adata,
            species=SPECIES,
            method='reference',
            do_filter=False,
            fuzzy_match=True,
            overwrite=True,
            copy=False,
        )
    print('Biotype annotation complete.')
    print(adata.var['biotype_category'].value_counts())
else:
    print('Gene biotype annotation skipped (RUN_BIOTYPE_ANNOTATION = False).')


### 3.2 Normalization & `.raw` Setup**Critical ordering**: Normalize first, then set `.raw`, then optionally filter biotypes.- `normalize_data()` — library-size normalization (log1p after size-factor scaling)- `.raw` — stores the **full-gene normalized expression** for downstream differential  expression testing and marker inspection- Quality-aware normalization: when `USE_QUALITY_AWARE_NORM = True`, normalization  adapts to per-cell QC scores, down-weighting low-quality cellsWhy `.raw` before biotype filtering:- Differential expression and marker gene inspection need access to all genes,  not just the protein-coding subset used for clustering.- Setting `.raw` before filtering preserves this full-gene reference.

In [ ]:
print("\n##====== 1. Normalizing Data & Setting .raw ======##")

if USE_QUALITY_AWARE_NORM:
    print("Using quality-aware normalization...")
    adata = scl.pp.quality_aware_normalize(
        adata,
        quality_metrics=["pct_counts_mt", "doublet_score"],
        input_layer="counts",
        output_layer="normalized",
        target_sum=NORM_TARGET_SUM,
    )
else:
    adata = scl.pp.normalize_data(
        adata,
        config=scl.pp.NormalizationConfig(
            method=NORM_METHOD,
            target_sum=NORM_TARGET_SUM,
            exclude_highly_expressed=EXCLUDE_HIGHLY_EXPRESSED,
            save_dir=str(RESULTS_DIR / "pp_normalization"),
        ),
    )

# Store full-gene normalized expression in .raw BEFORE any gene filtering.
# This is essential for downstream DE and marker inspection.
print("Storing normalized full-gene expression in adata.raw before biotype filtering...")
adata.raw = adata.copy()
print(f"  adata.raw shape: {adata.raw.shape}")


### 3.3 Normalization DiagnosticsVisualize the effect of normalization: before vs. after distributions,mean-variance relationships, and library-size normalization confirmation.

In [ ]:
if SHOW_NORMALIZATION_DIAGNOSTICS:
    try:
        scl.pp.plot_normalization_effect(
            adata,
            original_layer="counts",
            normalized_layer="normalized" if "normalized" in adata.layers else None,
            save_dir=str(RESULTS_DIR / "pp_normalization"),
        )
        print("Normalization diagnostics generated.")
    except Exception as e:
        print(f"Normalization diagnostics skipped: {e}")
else:
    print("Normalization diagnostics skipped (SHOW_NORMALIZATION_DIAGNOSTICS = False).")


### 3.4 Gene Biotype Filtering (Optional)Filter genes to retain only the specified biotypes (default: protein_coding).This is applied AFTER `.raw` is set, so `.raw` retains all genes for downstream DE.If `FILTER_BIOTYPE_AFTER_RAW = False`, this step is skipped.

In [ ]:
if FILTER_BIOTYPE_AFTER_RAW and RUN_BIOTYPE_ANNOTATION:
    print('Filtering genes by biotype...')
    adata = scl.pp.filter_genes_by_biotype(
        adata,
        keep_biotypes=KEEP_BIOTYPES,
        copy=True,
    )
    print(f'After biotype filtering: {adata.n_vars:,} genes remaining.')
    print(adata.var['biotype_category'].value_counts())
else:
    print('Biotype filtering skipped.')


### 3.5 Highly Variable Gene (HVG) SelectionTwo complementary methods, combined via union for multi-sample robustness:1. **scanpy (seurat flavor)** — standard HVG based on mean-variance relationship.   Good default, but can miss sample-specific signals in multi-sample data.2. **custom (sample-aware)** — identifies genes that are highly variable within   individual samples and highly expressed across samples. Better for   heterogeneous multi-sample designs.The `suggest_hvg_choice()` function guides the combination strategy.

In [ ]:
print('\n##====== 2. Highly Variable Gene (HVG) Selection ======##')

adata = scl.pp.find_hvgs(
    adata,
    input_layer='normalized',
    config=scl.pp.HVGConfig(
        method='scanpy',
        flavor=HVG_FLAVOR,
        n_top_genes=HVG_N_TOP_GENES_SCANPY,
        span=0.5,
    ),
)

adata = scl.pp.find_hvgs(
    adata,
    input_layer='normalized',
    config=scl.pp.HVGConfig(
        method='custom',
        sample_key=SAMPLE_KEY,
        n_top_genes=HVG_N_TOP_GENES_CUSTOM,
        min_n_samples=HVG_MIN_N_SAMPLES,
        n_highly_expressed_genes=50,
        n_specific_genes=10,
    ),
)

hvg_keys = [HVG_KEY_SCANPY, HVG_KEY_CUSTOM]
missing_hvg_keys = [k for k in hvg_keys if k not in adata.var.columns]
if missing_hvg_keys:
    raise KeyError(f'Expected HVG columns not found in adata.var: {missing_hvg_keys}')

print('\n--- HVG Set Comparison ---')
scl.pp.suggest_hvg_choice(
    adata,
    hvg_keys=hvg_keys,
    mode=HVG_UNION_MODE,
)


#### HVG Stability EvaluationBootstrapping-based stability assessment: resamples cells and recomputes HVGto measure how robust each gene's "highly variable" designation is.This is especially useful for:- Heterogeneous samples where HVG sets may be unstable- Publications where method justification is required- Deciding between union vs. intersection for combining HVG sets

In [ ]:
if RUN_HVG_STABILITY:
    print('\n--- HVG Stability Evaluation ---')
    try:
        stability_results = scl.pp.evaluate_hvg_stability(
            adata,
            hvg_key=HVG_KEY_SCANPY,
            n_bootstrap=20,
            flavor=HVG_FLAVOR,
            n_top_genes=HVG_N_TOP_GENES_SCANPY,
        )
        print('HVG stability evaluation complete.')
        if stability_results is not None:
            print(f'  HVG stability metrics stored in adata.uns[{repr(stability_results)}]')
    except Exception as e:
        print(f'HVG stability evaluation skipped: {e}')
else:
    print('HVG stability evaluation skipped (RUN_HVG_STABILITY = False).')


In [ ]:
# Select final HVG set.
adata = scl.pp.select_hvg_sets(
    adata,
    hvg_keys=hvg_keys,
    mode=HVG_UNION_MODE,
    subset=True,
    plot_venn=True,
    keep_raw=False,  # .raw was already set manually after normalization.
    save_dir=str(RESULTS_DIR / 'pp_hvg'),
)
print(f'\nFinal HVG set: {adata.n_vars:,} genes')


### 3.6 Regression, Scaling & PCASequential steps on the HVG subset:1. **Regression** — regress out technical covariates (total_counts, pct_mt, cell cycle scores)   from the normalized expression. This removes technical variation without distorting   biological signal.2. **Scaling** — z-score scaling to unit variance, capped at `max_value` to prevent   extreme values from dominating the PCA.3. **PCA** — dimensionality reduction to `min(50, n_cells-1, n_genes-1)` components.Note on cell cycle regression:If the biological process of interest may involve proliferation, consider runninga sensitivity analysis without cell cycle regression.

In [ ]:
print("\n##====== 3. Regression, Scaling & PCA ======##")# Regressionprint("Regressing out technical covariates...")regress_cfg = scl.pp.ScalingConfig(    vars_to_regress=VARS_TO_REGRESS,    regress_in_scale=REGRESS_IN_SCALE,    plot=False,    report=False,)adata = scl.pp.regress_out(    adata,    config=regress_cfg,    input_layer="normalized",    output_layer="regressed",)# Scalingadata.X = adata.layers["regressed"].copy()scale_cfg = scl.pp.ScalingConfig(    regress_in_scale=False,    max_value=SCALE_MAX_VALUE,    plot=False,    report=False,)adata = scl.pp.scale_data(adata, config=scale_cfg, output_layer="scaled")adata.X = adata.layers["scaled"].copy()# PCAn_pcs = min(N_PCS_MAX, adata.n_obs - 1, adata.n_vars - 1)sc.tl.pca(adata, n_comps=n_pcs)print(f"PCA completed: {n_pcs} components.")

### 3.7 Scaling DiagnosticsVisualize the effect of regression and scaling:- Before/after distributions of expression values- PCA elbow plot- Variance explained by technical vs. biological factors

In [ ]:
if SHOW_SCALING_DIAGNOSTICS:
    print("\n--- Scaling Diagnostics ---")
    try:
        scl.pp.plot_scaling_effect(
            adata,
            original_data="regressed",
            scaled_layer="scaled",
            save_dir=str(RESULTS_DIR / "pp_scaling"),
        )
        print("Scaling diagnostics generated.")
    except Exception as e:
        print(f"Scaling diagnostics skipped: {e}")
else:
    print("Scaling diagnostics skipped (SHOW_SCALING_DIAGNOSTICS = False).")


### 3.8 Diagnostic Embedding (Pre-Integration)Build a diagnostic UMAP using the unintegrated PCA to visualize sample-drivenstructure **before** any batch correction. This serves as a baseline:- If samples already mix well → integration may not be needed- If samples form distinct clusters → integration may help visualize shared biology- If samples separate by group → the batch and biology keys may be confounded

In [ ]:
print('\n##====== 4. Diagnostic Embedding (Pre-Integration) ======##')
optimization_results = scl.pp.optimize_neighbors_pcs(
    adata,
    config=scl.pp.NeighborsConfig(
        use_rep='X_pca',
        n_neighbors_list=N_NEIGHBORS_LIST,
        n_pcs_list=N_PCS_LIST,
        n_jobs=N_JOBS,
    ),
)
if len(optimization_results) > 0:
    best_params = optimization_results.loc[optimization_results['silhouette_score'].idxmax()]
    best_n_neighbors = int(best_params['n_neighbors'])
    best_n_pcs = int(best_params['n_pcs'])
    print(f'Diagnostic PCA embedding: n_neighbors={best_n_neighbors}, n_pcs={best_n_pcs}')
    sc.pp.neighbors(adata, use_rep='X_pca', n_neighbors=best_n_neighbors, n_pcs=best_n_pcs)
    sc.tl.umap(adata)
    adata.obsm['X_umap_pca'] = adata.obsm['X_umap'].copy()
else:
    print('No valid optimization result; using defaults n_neighbors=15, n_pcs=50.')
    sc.pp.neighbors(adata, use_rep='X_pca', n_neighbors=15, n_pcs=min(50, n_pcs))
    sc.tl.umap(adata)
    adata.obsm['X_umap_pca'] = adata.obsm['X_umap'].copy()


In [ ]:
# Plot diagnostic pre-integration UMAPplot_keys = [SAMPLE_KEY]if GROUP_KEY and GROUP_KEY in adata.obs.columns:    plot_keys.append(GROUP_KEY)palette = build_color_palette(plot_keys)sc.pl.embedding(    adata,    basis="X_umap_pca",    color=plot_keys,    title=[f"Pre-Integration: {k}" for k in plot_keys],    palette=palette,    ncols=2,    frameon="small",    show=False,    wspace=0.45,)print("Diagnostic pre-integration UMAP plotted (X_umap_pca).")

### 3.9 Integration Decision & Batch Correction**Auto-detection logic**:- If `RUN_INTEGRATION = "auto"` → detect confounding between batch key and biology columns- If batch is one-to-one with a biology column → integration is **skipped** to avoid  removing the biological signal of interest- If batch is independent of biology → integration proceedsWhen integration is run, the result is stored in a named key (e.g. `X_harmony_sampleID`)to distinguish it from unintegrated `X_pca`. The **final analysis graph** should usethe representation that preserves biological signal.

In [ ]:
print('\n##====== 5. Integration Decision ======##')
run_integration, integration_warnings = resolve_integration_flag(
    adata,
    INTEGRATION_BATCH_KEY,
    BIOLOGY_COLUMNS,
)
print(f"Integration batch key: {INTEGRATION_BATCH_KEY!r}")
print(f'Biology columns: {BIOLOGY_COLUMNS}')
print(f'Integration needed: {run_integration}')
if integration_warnings:
    print('\nIntegration warnings:')
    for w in integration_warnings:
        print(f'  ! {w}')

if run_integration:
    output_key = f'X_{INTEGRATION_METHOD}_{INTEGRATION_BATCH_KEY}'
    print(f"\nRunning {INTEGRATION_METHOD} integration on {INTEGRATION_BATCH_KEY!r}...")
    print(f'Output key: {output_key}')
    adata = scl.pp.batch_correction(
        adata,
        config=scl.pp.IntegrationConfig(
            method=INTEGRATION_METHOD,
            batch_key=INTEGRATION_BATCH_KEY,
            use_rep='X_pca',
            output_key=output_key,
        ),
        plot=False,
        force=True,
    )
    print(f"Integration complete. Embedding stored in adata.obsm[{output_key!r}].")
else:
    output_key = 'X_pca'
    print('\nSkipping integration; using unintegrated X_pca as final representation.')
    if integration_warnings:
        print('Reason(s): ' + '; '.join(integration_warnings))


### 3.10 Integration Quality EvaluationIf integration was run, evaluate its quality using established metrics:- **iLISI** (integration Local Inverse Simpson's Index) — measures batch mixing- **cLISI** (cell-type LISI) — measures cell-type separation- **ASW** (Average Silhouette Width) — batch-wise vs. cluster-wise- **kBet** — k-nearest neighbor batch effect testGood integration: high iLISI, high ASW_cluster, low ASW_batch.

In [ ]:
if EVALUATE_INTEGRATION and run_integration and output_key != 'X_pca':
    print('\n--- Integration Quality Evaluation ---')
    try:
        integration_metrics = scl.pp.evaluate_integration(
            adata,
            batch_key=INTEGRATION_BATCH_KEY,
            use_rep=output_key,
        )
        print('Integration quality metrics:')
        for metric_name, metric_value in integration_metrics.items():
            if isinstance(metric_value, (int, float)):
                print(f'  {metric_name}: {metric_value:.4f}')
            else:
                print(f'  {metric_name}: {metric_value}')
    except Exception as e:
        print(f'Integration evaluation skipped: {e}')
elif not run_integration:
    print('Skipping integration evaluation; no integration was run.')
else:
    print('Skipping integration evaluation (EVALUATE_INTEGRATION = False).')


### 3.11 Final Neighbors & UMAP OptimizationOptimize neighbors and UMAP on the final representation (integrated or unintegrated).If embedding optimization is enabled, scans over the parameter grid and selectsthe combination with the highest silhouette score.

In [ ]:
print('\n##====== 6. Final Graph & UMAP ======##')
print(f'Final representation: {output_key}')
if RUN_EMBEDDING_OPTIMIZATION:
    print('Running embedding optimization...')
    optimization_results = scl.pp.optimize_neighbors_pcs(
        adata,
        config=scl.pp.NeighborsConfig(
            use_rep=output_key,
            n_neighbors_list=N_NEIGHBORS_LIST,
            n_pcs_list=N_PCS_LIST,
            n_jobs=N_JOBS,
        ),
    )
    if len(optimization_results) > 0:
        best_params = optimization_results.loc[optimization_results['silhouette_score'].idxmax()]
        best_n_neighbors = int(best_params['n_neighbors'])
        best_n_pcs = int(best_params['n_pcs'])
        print(
            f"Optimized: n_neighbors={best_n_neighbors}, n_pcs={best_n_pcs} "
            f"(silhouette={best_params['silhouette_score']:.4f})"
        )
        display(optimization_results)
    else:
        print('No valid optimization result; using defaults.')
        best_n_neighbors = 15
        best_n_pcs = min(50, n_pcs)
else:
    best_n_neighbors = 15
    best_n_pcs = min(50, n_pcs)
    print(f'Using default parameters: n_neighbors={best_n_neighbors}, n_pcs={best_n_pcs}')

sc.pp.neighbors(adata, use_rep=output_key, n_neighbors=best_n_neighbors, n_pcs=best_n_pcs)
sc.tl.umap(adata, min_dist=0.3)

final_umap_key = f"X_umap_{output_key.replace('X_', '').lower()}"
adata.obsm[final_umap_key] = adata.obsm['X_umap'].copy()
print(f"Final UMAP saved as adata.obsm[{final_umap_key!r}]")


In [ ]:
print('\n##====== 7. Plotting Final Embedding ======##')
plot_keys = [SAMPLE_KEY]
if GROUP_KEY and GROUP_KEY in adata.obs.columns:
    plot_keys.append(GROUP_KEY)
if BATCH_KEY and BATCH_KEY in adata.obs.columns:
    plot_keys.append(BATCH_KEY)

palette = build_color_palette(plot_keys)
sc.pl.embedding(
    adata,
    basis=final_umap_key,
    color=plot_keys,
    title=[f'Final: {k}' for k in plot_keys],
    wspace=0.4,
    palette=palette,
    show=True,
)


---
## **Step 4: Contract Validation & Final Audit**

This section writes the same benchmark-grade scLucid audit fields used by workflow runs, while preserving the advanced notebook's manual decisions. It then validates only the stages produced by this notebook:

1. **Layer consistency** - required layers are present (`counts`, `normalized`, `regressed`, `scaled`).
2. **`.raw` integrity** - `.raw` should retain normalized full-gene expression before HVG/biotype filtering.
3. **Embedding keys** - expected embeddings are present.
4. **Stage contracts** - QC and preprocessing review summaries, configs, lineage, artifacts, module maturity, and compact audit summaries are present under `adata.uns["sclucid"]`.
5. **Step evidence** - preprocessing records per-step inputs, outputs, parameters, and review flags in `step_evidence_summary`.

If validation fails, treat it as provenance or object-contract debt to fix before promoting the notebook to a reusable project template.


In [ ]:
print('\n##====== 8. Final Object Audit ======##')
print(f'Final shape: {adata.shape}')
print(f'Layers: {list(adata.layers.keys())}')
print(f"Raw shape: {adata.raw.shape if adata.raw is not None else 'MISSING - DE and marker inspection may be compromised'}")
print(f'obsm keys: {list(adata.obsm.keys())}')
print(f"sclucid uns keys: {list(adata.uns.get('sclucid', {}).keys())}")

# The workflow contract expects a canonical HVG column. The advanced notebook may
# build a union/intersection mask first, so mirror the final selected set here.
if 'highly_variable_selected' in adata.var.columns:
    adata.var['highly_variable'] = adata.var['highly_variable_selected'].astype(bool)
elif HVG_KEY_CUSTOM in adata.var.columns:
    adata.var['highly_variable'] = adata.var[HVG_KEY_CUSTOM].astype(bool)
elif HVG_KEY_SCANPY in adata.var.columns:
    adata.var['highly_variable'] = adata.var[HVG_KEY_SCANPY].astype(bool)

required_layers = {'counts', 'normalized', 'regressed', 'scaled'}
missing_layers = required_layers - set(adata.layers.keys())
if missing_layers:
    print(f'WARNING: Missing expected layers: {missing_layers}')
else:
    print('All required layers present.')

raw_warning = None
if adata.raw is None:
    raw_warning = 'adata.raw is None; DE testing will be limited to HVG subset only.'
    print(f'WARNING: {raw_warning}')
elif adata.raw.shape[1] <= adata.shape[1]:
    raw_warning = 'adata.raw has <= genes than filtered adata; raw should retain all normalized genes before HVG/biotype filtering.'
    print(f'WARNING: {raw_warning}')

if 'X_harmony' in adata.obsm and output_key != 'X_harmony':
    print(f"NOTE: Generic 'X_harmony' found but final representation is '{output_key}'. Plotting should use '{final_umap_key}'.")

notebook_preprocess_steps = [
    'annotate_gene_biotypes' if RUN_BIOTYPE_ANNOTATION else 'skip_gene_biotype_annotation',
    'normalize_data' if not USE_QUALITY_AWARE_NORM else 'quality_aware_normalize',
    'set_raw',
    'filter_genes_by_biotype' if FILTER_BIOTYPE_AFTER_RAW and RUN_BIOTYPE_ANNOTATION else 'skip_biotype_filtering',
    'find_hvgs_scanpy',
    'find_hvgs_custom',
    'select_hvg_sets',
    'regress_out',
    'scale_data',
    'pca',
    'diagnostic_neighbors_umap',
    'batch_correction' if run_integration else 'skip_batch_correction',
    'final_neighbors_umap',
]
canonical_preprocess_steps = [
    'normalization',
    'set_raw',
    'hvg_selection',
    'subset_hvg',
    'regression',
    'scaling',
    'pca',
    'batch_correction',
    'neighbors_umap',
]
notebook_to_contract_steps = {
    'normalize_data': 'normalization',
    'quality_aware_normalize': 'normalization',
    'set_raw': 'set_raw',
    'find_hvgs_scanpy': 'hvg_selection',
    'find_hvgs_custom': 'hvg_selection',
    'select_hvg_sets': 'subset_hvg',
    'regress_out': 'regression',
    'scale_data': 'scaling',
    'pca': 'pca',
    'batch_correction': 'batch_correction',
    'skip_batch_correction': 'batch_correction',
    'final_neighbors_umap': 'neighbors_umap',
}

pp_review_config = scl.pp.WorkflowConfig(
    normalization=scl.pp.NormalizationConfig(
        method=NORM_METHOD,
        target_sum=NORM_TARGET_SUM,
        exclude_highly_expressed=EXCLUDE_HIGHLY_EXPRESSED,
        input_layer='counts',
        output_layer='normalized',
    ),
    hvg=scl.pp.HVGConfig(
        method='custom',
        flavor=HVG_FLAVOR,
        n_top_genes=HVG_N_TOP_GENES_CUSTOM,
        sample_key=SAMPLE_KEY,
        min_n_samples=HVG_MIN_N_SAMPLES,
    ),
    scaling=scl.pp.ScalingConfig(
        vars_to_regress=VARS_TO_REGRESS,
        regress_in_scale=REGRESS_IN_SCALE,
        max_value=SCALE_MAX_VALUE,
    ),
    integration=scl.pp.IntegrationConfig(
        method=INTEGRATION_METHOD if run_integration else None,
        batch_key=INTEGRATION_BATCH_KEY if run_integration else None,
        use_rep='X_pca',
        output_key=output_key if run_integration else None,
    ),
    graph=scl.pp.GraphConfig(n_pcs=int(best_n_pcs), n_neighbors=int(best_n_neighbors)),
    run_regression=bool(VARS_TO_REGRESS),
    run_integration=bool(run_integration),
    run_neighbors=True,
)

# Make the final HVG decision discoverable by the preprocess trace helpers.
pp_ns = adata.uns.setdefault('sclucid', {}).setdefault('preprocess', {})
pp_ns.setdefault('hvg', {}).update({
    'output_key': 'highly_variable',
    'input_layer': 'normalized',
    'method': f'{HVG_UNION_MODE}_of_scanpy_and_custom',
    'n_hvg': int(adata.var['highly_variable'].sum()) if 'highly_variable' in adata.var else None,
    'source_keys': list(hvg_keys),
})

preprocess_summary = {
    'workflow_layer': WORKFLOW_LAYER,
    'notebook_steps_executed': notebook_preprocess_steps,
    'notebook_to_contract_steps': notebook_to_contract_steps,
    'layer_summary': {
        'layers': list(adata.layers.keys()),
        'raw_shape': list(adata.raw.shape) if adata.raw is not None else None,
        'obsm_keys': list(adata.obsm.keys()),
    },
    'hvg_summary': {
        'hvg_keys': hvg_keys,
        'canonical_hvg_key': 'highly_variable',
        'mode': HVG_UNION_MODE,
        'n_final_genes': int(adata.n_vars),
    },
    'representation_summary': {
        'pca_key': 'X_pca',
        'final_representation': output_key,
        'final_umap_key': final_umap_key,
        'best_n_neighbors': int(best_n_neighbors),
        'best_n_pcs': int(best_n_pcs),
        'integration_ran': bool(run_integration),
        'integration_warnings': integration_warnings,
    },
    'warnings': [raw_warning] if raw_warning else [],
}

from scLucid.preprocess.trace import enrich_preprocessing_review_summary, validate_preprocessing_review_summary

preprocess_summary = enrich_preprocessing_review_summary(
    preprocess_summary,
    adata=adata,
    config=pp_review_config,
    successful_steps=canonical_preprocess_steps,
    tissue_type=TISSUE_TYPE,
    keep_intermediate_layers=True,
)

preprocess_review_summary = finalize_manual_review_summary(
    adata,
    module='preprocess',
    workflow_name='notebook_preprocess_advanced',
    steps=canonical_preprocess_steps,
    config=build_config_snapshot()['preprocess'],
    summary=preprocess_summary,
    save_dir=RESULTS_DIR / 'preprocess_review',
)
preprocess_specific_errors = validate_preprocessing_review_summary(preprocess_review_summary)
if preprocess_specific_errors:
    print('Preprocess review-summary warnings/errors:')
    for err in preprocess_specific_errors:
        print(f'  - {err}')

pp_validation = scl.pp.validate_preprocess_module_completeness(adata)
pp_compact = scl.pp.summarize_preprocess_review_summary(preprocess_review_summary)
print('\n---- Preprocess Module Maturity ----')
print(f"Preprocess module valid: {pp_validation['valid']}")
print(f"Preprocess maturity: {pp_compact['maturity_status']}")
print(f"Preprocess readiness: {pp_compact['readiness_status']} ({pp_compact['readiness_score']})")
print(f"QC input available: {pp_compact['qc_input_available']}")
print(f"Step status counts: {pp_compact['step_status_counts']}")
print(f"Review-required steps: {pp_compact['review_required_steps']}")
if pp_validation['issues']:
    print('Preprocess validation issues:')
    for issue in pp_validation['issues']:
        print(f'  - {issue}')
if pp_validation['warnings']:
    print('Preprocess validation warnings:')
    for warning in pp_validation['warnings']:
        print(f'  ! {warning}')

try:
    from scLucid.utils import record_contract_result, validate_all_stage_contracts
    contract_results = validate_all_stage_contracts(adata, stages=VALIDATE_STAGES, when='output')
    print('\n=== Stage Contract Validation ===')
    for stage, result in contract_results.items():
        record_contract_result(adata, stage, result)
        status = 'PASS' if result.valid else 'FAIL'
        print(f'  {stage}: {status}')
        if result.errors:
            for err in result.errors:
                print(f'    - {err}')
        if result.warnings:
            for warn in result.warnings:
                print(f'    ! {warn}')
    if any(not result.valid for result in contract_results.values()):
        print('\nNOTE: Contract failures should be fixed before reusing this notebook as a template.')
except Exception as exc:
    print(f'Contract validation skipped due to error: {exc}')

print(f"\n{'=' * 60}")
print('Preprocessing complete.')
print(f'  Cells:  {adata.n_obs:,}')
print(f'  Genes:  {adata.n_vars:,}')
print(f'  Final embedding: {final_umap_key}')
print(f"{'=' * 60}")


In [ ]:
# Save final preprocessed AnnData.
from scLucid.utils import record_artifact

output_path = DATA_DIR / 'Step2-sce_preprocessed.h5ad'
record_artifact(
    adata,
    'preprocess',
    'preprocessed_h5ad',
    str(output_path),
    kind='h5ad',
    description='Final preprocessed AnnData object',
)
adata.write_h5ad(output_path, compression='gzip')
print(f'\nSaved: {output_path}')
print('\n---- Preprocessing Complete ----')
